# Fraud detection – CatBoost submission
CatBoost on its own, taken out of `fraud_candidates.ipynb`. It uses the same features, model settings, CV folds, threshold rule and score mapping, so it writes the same `submission_catboost.csv` (`id,prediction`).

`prediction` is a fraud score in [0,1]: its ranking gives PR-AUC, and scores ≥ 0.5 are the fraud flags.

In [ ]:
%pip install -q catboost

import os, sys
import numpy as np
import pandas as pd

# Read from the repo's data/ folder if it exists; otherwise open an upload button (Colab)
def load_csv(name):
    for path in [f"../data/{name}", f"data/{name}", name]:
        if os.path.exists(path):
            return pd.read_csv(path)
    from google.colab import files
    print(f"Upload: {name}")
    return pd.read_csv(list(files.upload().keys())[0])

df = load_csv("Track 2 Training Dataset.csv")
test_df = load_csv("Track 2 Testing Dataset.csv")
print("Train:", df.shape, " Test:", test_df.shape)

In [ ]:
target_column = "fraud"
CATS = ["merchant_category", "country", "transaction_channel"]

# Adds per-row features (uses only that row's own values, so no label or cross-row leakage)
def engineer(d):
    d = d.copy()
    d["log_amt"] = np.log1p(d.transaction_amount)
    d["log_spend"] = np.log1p(d.spend_last_24h)
    d["log_age"] = np.log1p(d.account_age)
    d["hour_sin"] = np.sin(2 * np.pi * d.transaction_hour / 24)
    d["hour_cos"] = np.cos(2 * np.pi * d.transaction_hour / 24)
    d["night"] = d.transaction_hour.isin([23, 0, 1, 2, 3, 4]).astype(int)
    d["burst"] = (d.transactions_last_1h >= 4).astype(float)
    d["high24"] = (d.transactions_last_24h >= 11).astype(float)
    d["young"] = (d.account_age <= 30).astype(float)
    d["amt_to_avg_ticket"] = d.log_amt - np.log1p(d.spend_last_24h / d.transactions_last_24h.clip(lower=1))
    d["share_1h"] = d.transactions_last_1h / d.transactions_last_24h.clip(lower=1)
    d["risky_merchant"] = d.merchant_category.isin(["luxury", "cash_transfer", "electronics"]).astype(int)
    d["night_x_device"] = d.night * d.new_device
    d["n_flags"] = d[["burst", "high24", "young", "new_device", "night", "risky_merchant"]].fillna(0).sum(axis=1)
    for c in CATS:
        d[c] = d[c].fillna("Unknown").astype("category")
    return d

# --- DROP UNNECESSARY ID COLUMNS --- (id is a random label; checked: it carries no fraud signal)
X = engineer(df.drop(columns=["id", target_column]))
y = df[target_column].values
NUM_COLS = [c for c in X.columns if c not in CATS]

# Test set: same features, and exactly the same category lists as training
X_test = engineer(test_df.drop(columns=["id"]))
for c in CATS:
    X_test[c] = pd.Categorical(X_test[c].astype(str), categories=X[c].cat.categories)
X_test = X_test[X.columns]
print(f"{X.shape[1]} features, fraud rate {y.mean():.2%}")

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer
from catboost import CatBoostClassifier

def make_catboost():
    # Strongest single model in CV (ordered target statistics for categoricals, small trees, strong L2).
    # CatBoost needs categoricals as strings.
    to_str = FunctionTransformer(lambda d: d.astype({c: str for c in CATS}))
    return make_pipeline(to_str, CatBoostClassifier(iterations=800, depth=4, learning_rate=0.03, l2_leaf_reg=10,
                                                    cat_features=CATS, random_seed=0, verbose=0,
                                                    allow_writing_files=False))

# Probability of the fraud class
def p_fraud(model, X_):
    return model.predict_proba(X_)[:, list(model.classes_).index(1)]

# Percentile rank of the score (as in fraud_candidates.ipynb, where single models went through the rank blend).
# Same ordering as the probabilities, so PR-AUC is unchanged; kept so the submission matches exactly.
def rank_score(score):
    return pd.Series(score).rank(pct=True).values

In [ ]:
from sklearn.metrics import average_precision_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold

# 5-fold CV: every row is scored once by a model that never saw it (out-of-fold)
oof = np.zeros(len(y))
for fold, (tr, te) in enumerate(StratifiedKFold(n_splits=5, shuffle=True, random_state=0).split(X, y)):
    oof[te] = p_fraud(make_catboost().fit(X.iloc[tr], y[tr]), X.iloc[te])
    print(f"fold {fold} done")

# Threshold: PR-AUC does not depend on it (it only uses the ranking). It sets the 0/1 fraud flags scored by F1 / Recall.
# For every cut-off (share of rows flagged) build the confusion matrix; pick the share with the best (smoothed) F1.
def pick_share(score):
    rows = []
    for share in np.arange(0.002, 0.0601, 0.0005):
        flag = (score >= np.quantile(score, 1 - share)).astype(int)
        tn, fp, fn, tp = confusion_matrix(y, flag).ravel()
        rows.append({"share": share, "precision": tp / max(tp + fp, 1), "recall": tp / (tp + fn), "F1": 2 * tp / (2 * tp + fp + fn)})
    cm = pd.DataFrame(rows)
    cm["F1_smooth"] = cm["F1"].rolling(5, center=True, min_periods=1).mean()   # don't pick a one-off noisy spike
    return cm.loc[cm["F1_smooth"].idxmax()]

score = rank_score(oof)
best = pick_share(score)
share = best["share"]
pd.DataFrame([{"candidate": "catboost", "PR-AUC": average_precision_score(y, score), "flag share": best["share"],
               "precision": best["precision"], "recall": best["recall"], "F1": best["F1"]}]).set_index("candidate").round(4)

In [ ]:
# Maps scores to [0,1] without changing their order (PR-AUC unchanged) and puts the flag threshold at 0.5
def to_submission_score(score, thr):
    score = np.asarray(score, dtype=float)
    lo, hi = score.min(), score.max()
    below = 0.49 * (score - lo) / max(thr - lo, 1e-12)
    above = 0.51 + 0.49 * (score - thr) / max(hi - thr, 1e-12)
    return np.clip(np.where(score >= thr, above, below), 0, 1)

# Train once on all 20,000 training rows and score the test set
model = make_catboost().fit(X, y)
score = rank_score(p_fraud(model, X_test))
prob = to_submission_score(score, np.quantile(score, 1 - share))
sub = pd.DataFrame({"id": test_df["id"], "prediction": prob})
assert len(sub) == len(test_df) and sub["id"].is_unique
assert sub["prediction"].between(0, 1).all() and sub["prediction"].nunique() > 2
sub.to_csv("submission_catboost.csv", index=False)
print(f"submission_catboost.csv: {len(sub)} rows, flagged (>= 0.5): {(prob >= 0.5).sum()}")

# In Colab: download the file
if "google.colab" in sys.modules:
    from google.colab import files
    files.download("submission_catboost.csv")